In [1]:
import os
import csv
from datetime import datetime
import pandas as pd
import numpy as np
import random
from sklearn.metrics.pairwise import cosine_similarity

from evaluation.split import leave_last_out_split
from evaluation.metrics import recall_at_k, ndcg_at_k, catalog_coverage_at_k, novelty_at_k
from models.baselines import PopularityRecommender, RandomRecommender, TrendingRecommender
from models.knn import ItemItemKNN
from models.mf import BiasedMatrixFactorization, BPRMatrixFactorization

# --- GLOBAL SETTINGS ---
np.random.seed(42)
random.seed(42)
K = 20

print("=== PHASE 1: DATA PIPELINE ===")
df = pd.read_csv('../data/raw/dataset_TSMC2014_NYC.csv')

if len(df.columns) == 8:
    df.columns =['user_id', 'venue_id', 'venue_category_id', 'venue_category_name',
                  'latitude', 'longitude', 'timezone_offset', 'date_time']

df['date_time'] = pd.to_datetime(df['date_time'])

# Semantic Filtering
boring_categories =['Home', 'Private', 'Office', 'Subway', 'Train Station', 'Gym', 'Fitness', 'Bus', 'Airport', 'Bank', 'Laundromat', 'Grocery', 'Pharmacy']
pattern = '|'.join(boring_categories)
df_travel = df[~df['venue_category_name'].str.contains(pattern, case=False, na=False)].copy()

# Enforce minimum interactions for temporal split
user_counts = df_travel['user_id'].value_counts()
valid_users = user_counts[user_counts >= 2].index
df_travel = df_travel[df_travel['user_id'].isin(valid_users)].copy()

print(f"Data filtered. Remaining interactions: {len(df_travel)}")

train_df, test_df = leave_last_out_split(df_travel, user_col='user_id', time_col='date_time')

# Precompute data for downstream tasks
train_item_counts = train_df['venue_id'].value_counts().to_dict()
total_train_interactions = len(train_df)
total_items = train_df['venue_id'].nunique()
venue_names = df.drop_duplicates('venue_id').set_index('venue_id')['venue_category_name'].to_dict()


print("\n=== PHASE 2: MODEL TRAINING ===")

# Baselines (Week 1)
pop_model = PopularityRecommender()
pop_model.fit(train_df, item_col='venue_id')

trend_model = TrendingRecommender(days_window=30)
trend_model.fit(train_df, item_col='venue_id', time_col='date_time')

rand_model = RandomRecommender()
rand_model.fit(train_df, item_col='venue_id')

# Collaborative Filtering (Weeks 3 & 4)
knn_model = ItemItemKNN(k_neighbors=100, shrinkage=10)
knn_model.fit(train_df, item_col='venue_id')

mf_model = BiasedMatrixFactorization(k_factors=16, epochs=5)
mf_model.fit(train_df, item_col='venue_id')

bpr_model = BPRMatrixFactorization(k_factors=32, epochs=10)
bpr_model.fit(train_df, item_col='venue_id')

print("\nAll models trained and ready for evaluation.")

=== PHASE 1: DATA PIPELINE ===


C:\Users\pssol\AppData\Local\Temp\ipykernel_20528\3413466382.py:27: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date_time'] = pd.to_datetime(df['date_time'])


Data filtered. Remaining interactions: 160704
Sorting data chronologically to prevent time leakage...
Splitting data: Leave-last-out...
Train Set: 159621 interactions
Test Set: 1083 interactions

=== PHASE 2: MODEL TRAINING ===
Training Popularity Baseline...
Learned popularity ranking for 31230 unique items.
Training Trending Baseline (Last 30 days)...
Learned 4681 trending items.
Training Random Baseline...
Training Item-Item kNN (k=100, shrinkage=10)...
Normalizing vectors and computing Cosine Similarity...
Applying shrinkage to downweight coincidental overlaps...
Pruning to Top-K neighborhoods to reduce noise...
kNN Training complete!
Training Biased MF (k=16, epochs=5)...
Epoch 1/5 completed.
Epoch 2/5 completed.
Epoch 3/5 completed.
Epoch 4/5 completed.
Epoch 5/5 completed.
MF training complete!
Training BPR MF (k=32, epochs=10)...
Epoch 1/10 completed.
Epoch 2/10 completed.
Epoch 3/10 completed.
Epoch 4/10 completed.
Epoch 5/10 completed.
Epoch 6/10 completed.
Epoch 7/10 complet

In [2]:
print("\n=== PHASE 3: MASTER EVALUATION LOOP ===")

# Storage arrays for all models
results = {
    'Popularity': {'recs': [], 'ndcgs': [], 'recalls': [], 'novelties':[], 'all_recs': []},
    'Trending (30d)': {'recs':[], 'ndcgs': [], 'recalls': [], 'novelties': [], 'all_recs':[]},
    'Random': {'recs': [], 'ndcgs': [], 'recalls': [], 'novelties':[], 'all_recs': []},
    'Item-Item kNN': {'recs':[], 'ndcgs': [], 'recalls': [], 'novelties': [], 'all_recs':[]},
    'Biased MF (SQ)': {'recs': [], 'ndcgs': [], 'recalls': [], 'novelties':[], 'all_recs': []},
    'BPR MF': {'recs': [], 'ndcgs': [], 'recalls':[], 'novelties': [], 'all_recs':[]}
}

models = {
    'Popularity': pop_model,
    'Trending (30d)': trend_model,
    'Random': rand_model,
    'Item-Item kNN': knn_model,
    'Biased MF (SQ)': mf_model,
    'BPR MF': bpr_model
}

print("Bereite User-Historien vor...")
train_history = train_df.groupby('user_id')['venue_id'].apply(set).to_dict()

test_users = test_df['user_id'].unique()
print(f"Evaluiere {len(test_users)} User. Das kann ein paar Sekunden dauern...")

for user in test_users:
    true_item = test_df[test_df['user_id'] == user]['venue_id'].iloc[0]
    user_hist = train_history.get(user, set())

    # Generate, score, and store for every model in one pass
    for name, model in models.items():
        recs = model.recommend(user, user_history=user_hist, k=K)

        results[name]['all_recs'].append(recs)
        results[name]['recalls'].append(recall_at_k(recs, [true_item], k=K))
        results[name]['ndcgs'].append(ndcg_at_k(recs, [true_item], k=K))
        results[name]['novelties'].append(novelty_at_k(recs, train_item_counts, total_train_interactions, k=K))

print("\n--- 📊 FINAL MACRO-AVERAGED RESULTS TABLE ---")
print(f"{'Model':<16} | {'Recall@20':<10} | {'NDCG@20':<10} | {'Coverage@20':<12} | {'Novelty@20'}")
print("-" * 75)

for name in models.keys():
    cov = catalog_coverage_at_k(results[name]['all_recs'], total_items)
    mean_rec = np.mean(results[name]['recalls'])
    mean_ndcg = np.mean(results[name]['ndcgs'])
    mean_nov = np.mean(results[name]['novelties'])

    print(f"{name:<16} | {mean_rec:.4f}     | {mean_ndcg:.4f}     | {cov:.<10.4%} | {mean_nov:.4f}")

# Automated Sanity Check
if np.mean(results['Popularity']['recalls']) > 0.15:
    print("⚠️ WARNING: Popularity baseline is annoyingly strong (>15% Recall). Check for temporal leakage!")
else:
    print("✅ Sanity Checks Passed.")


print("\n=== PHASE 4: DIAGNOSTICS & EXPLAINABILITY ===")

# 4a. kNN Explainability (Week 3, Slide 45)
print("\n--- 4A. kNN CONSTRUCTIVE EXPLAINABILITY ---")
heavy_users = train_df['user_id'].value_counts()
sample_user = heavy_users[heavy_users > 5].sample(1, random_state=42).index[0]
print(f"Generating explanations for User {sample_user}:")

explained_recs = knn_model.recommend(sample_user, user_history=train_history.get(sample_user, set()), k=3, return_explanations=True)
for i, (rec_id, explanation) in enumerate(explained_recs):
    name = venue_names.get(rec_id, "Unknown Venue")
    if "similar to" in explanation:
        hist_id = explanation.split(" ")[6]
        hist_name = venue_names.get(hist_id, hist_id)
        explanation = f"Recommended because it is similar to '{hist_name}' which you visited recently."
    print(f"   {i+1}. {name}\n      -> {explanation}")

# 4b. BPR Bias Inspection (Week 4, Slide 27)
print("\n--- 4B. BPR BIAS INSPECTION ---")
top_bias_indices = np.argsort(bpr_model.b_i)[-5:][::-1]
print("Top 5 Venues by Learned Bias (Should be famous/popular places):")
for idx in top_bias_indices:
    raw_id = bpr_model.reverse_item_mapping[idx]
    print(f"   -> {venue_names.get(raw_id, 'Unknown')} (Bias: {bpr_model.b_i[idx]:.4f})")

# 4c. BPR Latent Space Inspection (Week 4, Slide 27)
print("\n--- 4C. BPR LATENT SEMANTICS ---")
park_ids =[vid for vid, name in venue_names.items() if 'Park' in str(name)]
if park_ids:
    # Pick a popular park that the model likely learned well
    anchor_id =[vid for vid in park_ids if vid in bpr_model.item_mapping][0]
    anchor_idx = bpr_model.item_mapping[anchor_id]
    anchor_vector = bpr_model.Q[anchor_idx].reshape(1, -1)

    similarities = cosine_similarity(anchor_vector, bpr_model.Q)[0]
    nearest_indices = np.argsort(similarities)[-6:-1][::-1]

    print(f"Nearest Neighbors in Latent Space for '{venue_names.get(anchor_id, 'Unknown')}':")
    for idx in nearest_indices:
        raw_id = bpr_model.reverse_item_mapping[idx]
        print(f"   -> {venue_names.get(raw_id, 'Unknown')} (Latent Sim: {similarities[idx]:.4f})")


print("\n=== PHASE 5: EXPERIMENT LOGGING ===")
runs_file = '../runs/runs.csv'
os.makedirs('../runs', exist_ok=True)

# We log our current champion model: BPR
bpr_cov = catalog_coverage_at_k(results['BPR MF']['all_recs'], total_items)

run_data = {
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'dataset': 'Foursquare_NYC_TravelBuddy',
    'model': 'BPR_MF',
    'k_evaluated': K,
    'hyperparams': f'k_factors={bpr_model.k}, epochs={bpr_model.epochs}, reg={bpr_model.reg}',
    'recall': round(np.mean(results['BPR MF']['recalls']), 4),
    'ndcg': round(np.mean(results['BPR MF']['ndcgs']), 4),
    'coverage': round(bpr_cov, 4)
}

file_exists = os.path.isfile(runs_file)
with open(runs_file, mode='a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=run_data.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(run_data)

print(f"✅ BPR metrics successfully logged to {runs_file}!")
print("Notebook complete. Ready for handoff.")


=== PHASE 3: MASTER EVALUATION LOOP ===
Bereite User-Historien vor...
Evaluiere 1083 User. Das kann ein paar Sekunden dauern...

--- 📊 FINAL MACRO-AVERAGED RESULTS TABLE ---
Model            | Recall@20  | NDCG@20    | Coverage@20  | Novelty@20
---------------------------------------------------------------------------
Popularity       | 0.0065     | 0.0021     | 0.1025%... | 9.4646
Trending (30d)   | 0.0037     | 0.0012     | 0.0768%... | 10.9278
Random           | 0.0009     | 0.0002     | 50.2594%.. | 16.0136
Item-Item kNN    | 0.0028     | 0.0010     | 16.2024%.. | 13.3739
Biased MF (SQ)   | 0.0018     | 0.0005     | 0.4035%... | 10.0156
BPR MF           | 0.0166     | 0.0048     | 2.0301%... | 10.2011
✅ Sanity Checks Passed.

=== PHASE 4: DIAGNOSTICS & EXPLAINABILITY ===

--- 4A. kNN CONSTRUCTIVE EXPLAINABILITY ---
Generating explanations for User 745:
   1. French Restaurant
      -> Recommended because it is similar to 'Music Venue' which you visited recently.
   2. Bar
      -